# 2025-06-18. Network-features

## Summary

- Building matrices for analysis
- Addressing the scaling issue of modularity calculations
- Displaying network features

## Pre-processing

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
habitat_palette = {
    "Crop": "#0CAB7B", 
    "Edge": "#3A8DFA", 
    "Oak": "#FB2231",
    "Wasteland": "#FFC51F"
}
import tqdm
from scipy.stats import linregress
import matplotlib.pyplot as plt

## Ecological network structure

In [ ]:
motus_df.value_counts(
    ['Host_taxon', 'species']
).reset_index().pivot(
    index='Host_taxon', columns='species', 
    values='count'
).fillna(0.0).to_csv(
    "../results/2025-06-13.sprint/intdata.adj-mat.full.host-bacteria.csv", sep=';')

In [ ]:
for i in range(10):
    motus_df.value_counts(
        ['Host_taxon', 'species']
    ).reset_index().sample(n=10).pivot(
        index='Host_taxon', columns='species', 
        values='count'
    ).fillna(0.0).to_csv(
        f"../results/2025-06-13.sprint/modularity-test/intdatan10.{i:03d}.adj-mat.full.host-bacteria.csv", sep=';')
    
for i in range(10):
    motus_df.value_counts(
        ['Host_taxon', 'species']
    ).reset_index().sample(n=50).pivot(
        index='Host_taxon', columns='species', 
        values='count'
    ).fillna(0.0).to_csv(
        f"../results/2025-06-13.sprint/modularity-test/intdatan50.{i:03d}.adj-mat.full.host-bacteria.csv", sep=';')
    
for i in range(10):
    motus_df.value_counts(
        ['Host_taxon', 'species']
    ).reset_index().sample(n=100).pivot(
        index='Host_taxon', columns='species', 
        values='count'
    ).fillna(0.0).to_csv(
        f"../results/2025-06-13.sprint/modularity-test/intdatan100.{i:03d}.adj-mat.full.host-bacteria.csv", sep=';')

In [ ]:
for habitat in ['Crop', 'Oak', 'Edge', 'Wasteland']:
    motus_df.query(
        'Habitat == "{0}"'.format(habitat)
    ).value_counts(
        ['Host_taxon', 'species']
    ).reset_index().pivot(
        index='Host_taxon', columns='species', 
        values='count'
    ).fillna(0.0).to_csv(
        f"../results/2025-06-13.sprint/intdata.adj-mat.host.{habitat}.bacteria.csv", sep=';')

### Benchmark

In [ ]:
u

In [ ]:
benchmark = []
with open("../results/2025-06-13.sprint/modularity-test/.input", "r") as f:
    benchmark_files = list(map(lambda x: x.strip(), f.readlines()))
for file in benchmark_files:
    u = pd.read_csv("../results/2025-06-13.sprint/modularity-test/" + file, sep=',')
    level = file.split(".")[0]
    time = u.set_index('Metric').loc['ElapsedTime'].values[0]
    benchmark.append({"level": level, "time": time})
benchmark = pd.DataFrame.from_records(benchmark)
benchmark['size'] = benchmark['level'].apply(lambda x: int(x.replace("intdatan", "")))
benchmark['logtime'] = benchmark['time'].apply(lambda x: np.log10(x))
benchmark

In [ ]:
sns.regplot(benchmark, x='size', y='time', order=1)

In [ ]:
from scipy.stats import linregress

In [ ]:
reg = linregress(benchmark['size'], benchmark['time'])

In [ ]:
(reg.slope * 1490) + reg.intercept

## Results

In [ ]:
data = pd.read_csv("../results/2025-06-13.sprint/network-properties/.input", header=None, names=['observations_file'])
data['Habitat'] = data['observations_file'].apply(lambda x: x.split('.')[1])
data['null_file'] = data['observations_file'].apply(lambda x: x.replace("observed", "null"))
data

In [ ]:
observations = []
for _, line in data.iterrows():
    tmp = pd.read_csv(
        "/home/bcz/research/miripvir25/results/2025-06-13.sprint/network-properties/" + line.observations_file
    ).set_index('Metric').to_dict()['Value']
    tmp['Habitat'] = line['Habitat']
    observations.append(tmp)
observations = pd.DataFrame.from_records(observations)
observations

In [ ]:
null = []
for _, line in data.iterrows():
    tmp = pd.read_csv(
        "/home/bcz/research/miripvir25/results/2025-06-13.sprint/network-properties/" + line.null_file
    )
    tmp['Habitat'] = line['Habitat']
    null.append(tmp)
null = pd.concat(null)
null

In [ ]:
system = "Full"
null = null.query('Habitat == "{:s}"'.format(system))
observed = observations.query('Habitat == "{:s}"'.format(system))


In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(6, 3))
sns.kdeplot(null['connectance'], ax=ax[0], fill=True, color='teal', edgecolor='black', linewidth=0.65)
ax[0].axvline(observed.loc['connectance']['Value'], color='red', linestyle='--')
ax[0].text(observed.loc['connectance']['Value'], 2000, f"{observed.loc['connectance']['Value']:.4f}", color='red', fontsize=8, ha='center', va='center', bbox=dict(facecolor='white', edgecolor='white'))
sns.kdeplot(null['NODF'], ax=ax[1], fill=True, color='teal', edgecolor='black', linewidth=0.65)
ax[1].axvline(observed.loc['NODF']['Value'], color='red', linestyle='--')
ax[1].text(observed.loc['NODF']['Value'], 0.5, f"{observed.loc['NODF']['Value']:.4f}", color='red', fontsize=8, ha='center', va='center', bbox=dict(facecolor='white', edgecolor='white'))
sns.kdeplot(null['modularity Q'], ax=ax[2], fill=True, color='teal', edgecolor='black', linewidth=0.65)
ax[2].axvline(observed.loc['modularity Q']['Value'], color='red', linestyle='--')
ax[2].text(observed.loc['modularity Q']['Value'], 0.5, f"{observed.loc['modularity Q']['Value']:.4f}", color='red', fontsize=8, ha='center', va='center', bbox=dict(facecolor='white', edgecolor='white'))
fig.tight_layout()